In [17]:
from pathlib import Path 
from PIL import Image 
import pandas as pd 
import hashlib
from collections import Counter 

# Dataset locations 
DATASET_DIR = Path("../dataset/raw/Plant Village Dataset")

# Verify the path 
print("Dataset path:",DATASET_DIR.resolve())
print("Dataset exists:", DATASET_DIR.exists())

# Supported image formats
SUPPORTED_FORMATS = {".jpg",".jpeg",".png"}

#Dataset splits 
SPLITS = ["Train","Val","Test"]


Dataset path: A:\Crop-Disease-Detection\dataset\Raw\Plant Village Dataset
Dataset exists: True


In [18]:
image_records = []
unusual_files = []
empty_classes = []

# Check each dataset split
for split in SPLITS:

    split_path = DATASET_DIR / split

    if not split_path.exists():
        print(f"Warning: {split} folder not found!")
        continue

    # Check each disease class
    for class_folder in split_path.iterdir():

        if not class_folder.is_dir():
            continue

        # Find all files inside this class folder
        files = [f for f in class_folder.rglob("*") if f.is_file()]

        supported_files = [
            f for f in files
            if f.suffix.lower() in SUPPORTED_FORMATS
        ]

        if not supported_files:
            empty_classes.append({
                "Dataset": split,
                "Class": class_folder.name
            })

        for file_path in files:

            extension = file_path.suffix.lower()

            # Report unusual file formats
            if extension not in SUPPORTED_FORMATS:
                unusual_files.append({
                    "Dataset": split,
                    "Class": class_folder.name,
                    "File": str(file_path),
                    "Extension": extension or "[no extension]"
                })
                continue

            record = {
                "Dataset": split,
                "Class": class_folder.name,
                "File": str(file_path),
                "Extension": extension,
                "Width": None,
                "Height": None,
                "Mode": None,
                "Format": None,
                "Size_KB": round(file_path.stat().st_size / 1024, 2),
                "Status": "Valid",
                "Error": None,
                "SHA256": None
            }

            try:
                # First check whether the image is valid
                with Image.open(file_path) as img:
                    img.verify()

                # Reopen it to read image properties
                with Image.open(file_path) as img:
                    record["Width"] = img.width
                    record["Height"] = img.height
                    record["Mode"] = img.mode
                    record["Format"] = img.format

                # Calculate exact file hash
                file_hash = hashlib.sha256(
                    file_path.read_bytes()
                ).hexdigest()

                record["SHA256"] = file_hash

            except Exception as e:
                record["Status"] = "Corrupted"
                record["Error"] = str(e)

            image_records.append(record)

print("Image scanning completed!")

Image scanning completed!


In [19]:
# Convert results into DataFrames

quality_df = pd.DataFrame(image_records)
unusual_df= pd.DataFrame(unusual_files)
empty_df=pd.DataFrame(empty_classes)

print("Total Supported Images: ",len(quality_df))

print(
    "Valid Images: ",
    (quality_df["Status"] == "Valid").sum()
)

print(
    "Corrupted Images: ",
    (quality_df["Status"] == "Corruped").sum()
)

print("Unusual files: ",len(unusual_df))
print("Empty class folders: ",len(empty_df))

Total Supported Images:  67118
Valid Images:  67118
Corrupted Images:  0
Unusual files:  0
Empty class folders:  0


In [20]:
# Detect duplicate images

# Consider only valid images 
valid_df = quality_df[
    quality_df["Status"] == "Valid"
].copy()

# Find hashes that appears more than once 
duplicate_df = valid_df[
    valid_df.duplicated(subset="SHA256", keep=False)
].sort_values("SHA256")

print("Total duplicate file enteries: ",len(duplicate_df))

duplicate_df[
    ["Dataset", "Class", "File","SHA256"]
]


Total duplicate file enteries:  66


,Dataset,Class,File,SHA256
47157,Train,Tomato - Healthy,..\dataset\raw\Plant Village Dataset\Train\Tom...,12852b3616bcf0ba39b03b5677e74f50f5f6abf4d38957...
47697,Train,Tomato - Healthy,..\dataset\raw\Plant Village Dataset\Train\Tom...,12852b3616bcf0ba39b03b5677e74f50f5f6abf4d38957...
48679,Train,Tomato - Late Blight,..\dataset\raw\Plant Village Dataset\Train\Tom...,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...
48915,Train,Tomato - Late Blight,..\dataset\raw\Plant Village Dataset\Train\Tom...,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...
51731,Train,Tomato - Septoria Leaf Spot,..\dataset\raw\Plant Village Dataset\Train\Tom...,257c75912bb03678b9af067545101252aa59790950d9ad...
...,...,...,...,...
5897,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,ec2fe5801c82d7ffff332b9429480c6a432830ccce2f7e...
7470,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,f9658c2a55775a239cd5fecc3a42114ec84f46cd68aa73...
7287,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,f9658c2a55775a239cd5fecc3a42114ec84f46cd68aa73...
7162,Train,Apple - Healthy,..\dataset\raw\Plant Village Dataset\Train\App...,ffd85543367c06fcc7008315354b387dc9f995627a900c...


In [21]:
# Analyze Image Dimensions

dimension_counts= (
    valid_df.groupby(["Width","Height"])
    .size()
    .reset_index(name="Image_Count")
    .sort_values("Image_Count",ascending=False)
)

dimension_counts.head(20)


,Width,Height,Image_Count
0,256,256,67118


In [22]:
# To check smallest and largest images
print("Smallest width:", valid_df["Width"].min())
print("Largest width:", valid_df["Width"].max())

print("Smallest height:", valid_df["Height"].min())
print("Largest height:", valid_df["Height"].max())

Smallest width: 256
Largest width: 256
Smallest height: 256
Largest height: 256


In [23]:
# Analyze color models

mode_counts = valid_df["Mode"].value_counts()

print(mode_counts)

Mode
RGB    67118
Name: count, dtype: int64


In [24]:
# Check corrupted images and empty folders 

# Display corrupted images
corrupted_df = quality_df[
    quality_df["Status"] == "Corrupted"
]
corrupted_df[["Dataset","Class","File","Error"]]


,Dataset,Class,File,Error


In [31]:
# Find duplicate images across different dataset splits

cross_split_duplicates = (
    quality_df.groupby("SHA256")
    .filter(
        lambda group:
        group["Dataset"].nunique() > 1
        and group["SHA256"].notna().all()
    )
    .sort_values("SHA256")
    .reset_index(drop=True)
)

print("Cross-split duplicates found:", len(cross_split_duplicates))

Cross-split duplicates found: 14


In [34]:
cross_split_summary = (
    cross_split_duplicates
    .groupby("SHA256")
    .agg(
        Occurrences=("SHA256", "size"),
        Splits=("Dataset", "nunique"),
        Classes=("Class", "nunique"),
        Locations=("File", list)
    )
    .reset_index()
)

print("Unique duplicate groups:", len(cross_split_summary))
print("Total duplicate entries:", len(cross_split_duplicates))

display(cross_split_summary)

Unique duplicate groups: 7
Total duplicate entries: 14


,SHA256,Occurrences,Splits,Classes,Locations
0,4913a3835e692530e507d542caccacc0d0a7b2fd708782...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\To...
1,5d486b62abe59b8dea5aef6b63f887bacd12bbc3cb2a15...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\Ap...
2,61f6fa6195ef5c59c4918a292c641d4cd3d2d31ff051dc...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\To...
3,68cda8c3ceca7626cc6727e14186d356b10d1e345eb16e...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\To...
4,a05e573ae6c4a28b077b8c9d01249be849f4f8e1790b26...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\To...
5,addddf5cb2a74192c9836be70fc223659062675eb325d5...,2,2,1,[..\dataset\raw\Plant Village Dataset\Val\Toma...
6,ec2fe5801c82d7ffff332b9429480c6a432830ccce2f7e...,2,2,1,[..\dataset\raw\Plant Village Dataset\Train\Ap...


In [35]:
different_label_duplicates = cross_split_summary[
    cross_split_summary["Classes"] > 1
]

print(
    "Duplicate groups with different labels:",
    len(different_label_duplicates)
)

display(different_label_duplicates)

Duplicate groups with different labels: 0


,SHA256,Occurrences,Splits,Classes,Locations


In [36]:
for _, row in cross_split_summary.iterrows():
    print("\nOccurrences:", row["Occurrences"])
    print("Locations:")

    for location in row["Locations"]:
        print(" -", location)


Occurrences: 2
Locations:
 - ..\dataset\raw\Plant Village Dataset\Train\Tomato - Late Blight\5de6da85-f8c4-48c4-b463-3e6bd78884cc___GHLB_PS Leaf 24 Day 16.jpg
 - ..\dataset\raw\Plant Village Dataset\Val\Tomato - Late Blight\3fae9c64-18f0-4a67-9f97-554248bb1bed___GHLB_PS Leaf 24 Day 16.jpg

Occurrences: 2
Locations:
 - ..\dataset\raw\Plant Village Dataset\Train\Apple - Healthy\c21cf428-bfc3-4710-b5d2-69d1c0e94748___RS_HL 6268.JPG
 - ..\dataset\raw\Plant Village Dataset\Val\Apple - Healthy\14896dc0-688d-456f-b5ec-a037695b0193___RS_HL 6268.JPG

Occurrences: 2
Locations:
 - ..\dataset\raw\Plant Village Dataset\Train\Tomato - Healthy\cfd491d6-4af5-4728-8f0e-0d330a07174a___GH_HL Leaf 482.2.JPG
 - ..\dataset\raw\Plant Village Dataset\Val\Tomato - Healthy\1af0bfe1-4bcf-4b8b-be66-5d0953eb647e___GH_HL Leaf 482.2.JPG

Occurrences: 2
Locations:
 - ..\dataset\raw\Plant Village Dataset\Train\Tomato - Healthy\37aad83b-7ff8-4b35-b3ed-fb8e0f54910b___GH_HL Leaf 342.1.JPG
 - ..\dataset\raw\Plant Village

In [37]:
# Group all images by their SHA256 hash
all_duplicate_summary = (
    quality_df
    .dropna(subset=["SHA256"])
    .groupby("SHA256")
    .agg(
        Occurrences=("SHA256", "size"),
        Splits=("Dataset", "nunique"),
        Classes=("Class", "nunique"),
        Locations=("File", list)
    )
    .reset_index()
)

# Select duplicate groups that occur more than once
all_duplicate_summary = all_duplicate_summary[
    all_duplicate_summary["Occurrences"] > 1
]

# Select groups occurring within only one split
within_split_duplicates = all_duplicate_summary[
    all_duplicate_summary["Splits"] == 1
]

print("Total duplicate groups:",
      len(all_duplicate_summary))

print("Groups duplicated within the same split:",
      len(within_split_duplicates))

print("Duplicate entries within the same split:",
      within_split_duplicates["Occurrences"].sum())

display(within_split_duplicates)

Total duplicate groups: 33
Groups duplicated within the same split: 26
Duplicate entries within the same split: 52


,SHA256,Occurrences,Splits,Classes,Locations
4804,12852b3616bcf0ba39b03b5677e74f50f5f6abf4d38957...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
7563,1d202e8c6b21351c851a4220c6ac5be10c4bf7f659fa82...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
9727,257c75912bb03678b9af067545101252aa59790950d9ad...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
9890,262198126fd6d982126b69261723e13abb36b383138797...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\Ap...
10913,2a05a74db3a270ac0ec77c00ebe89a332eed5898194351...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
11156,2aeb7479e975df37dfb54aa7b403f57f2ad90b018c3780...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
13252,32e9ffabaa054c19ae8df95e94fbd7a25105defdea1214...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\To...
13571,34316dd6e29238d888396de01d9838b0c6f94ab2fa335f...,2,1,1,[..\dataset\raw\Plant Village Dataset\Test\Cor...
14374,372bc4a6e18895011eaa18638acf8cdd83db9c5909af02...,2,1,1,[..\dataset\raw\Plant Village Dataset\Train\Ap...
14446,375eee7b5da77c7afdd73d089574bee3148d04cbe266ee...,2,1,1,[..\dataset\raw\Plant Village Dataset\Val\Toma...


In [32]:
# To check the folders that contains no valid images

valid_class_counts = (
    valid_df.groupby(["Dataset", "Class"])
    .size()
)

for split in SPLITS:

    split_path = DATASET_DIR / split

    if not split_path.exists():
        continue

    for folder in split_path.iterdir():

        if not folder.is_dir():
            continue

        count = valid_class_counts.get(
            (split, folder.name), 0
        )

        if count == 0:
            print(
                f"No valid images: {split}/{folder.name}"
            )


In [38]:
# Check all duplicate groups for different class labels
label_conflicts = all_duplicate_summary[
    all_duplicate_summary["Classes"] > 1
]

print("Total duplicate groups:",
      len(all_duplicate_summary))

print("Groups with different labels:",
      len(label_conflicts))

print("Groups within the same split:",
      len(within_split_duplicates))

print("Groups across different splits:",
      len(cross_split_summary))

Total duplicate groups: 33
Groups with different labels: 0
Groups within the same split: 26
Groups across different splits: 7


In [33]:
# Save all quality reports 

# Create report folder
REPORT_DIR = Path("../dataset/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save reports
quality_df.to_csv(
    REPORT_DIR / "image_quality_report.csv",
    index=False
)

duplicate_df.to_csv(
    REPORT_DIR / "duplicate_images.csv",
    index=False
)

cross_split_duplicates.to_csv(
    REPORT_DIR / "cross_split_duplicates.csv",
    index=False
)

unusual_df.to_csv(
    REPORT_DIR / "unusual_files.csv",
    index=False
)

empty_df.to_csv(
    REPORT_DIR / "empty_classes.csv",
    index=False
)

dimension_counts.to_csv(
    REPORT_DIR / "image_dimensions.csv",
    index=False
)

print("All quality reports saved successfully!")

All quality reports saved successfully!


In [39]:
from pathlib import Path

# Define the cleaned dataset directory
CLEANED_DIR = Path(
    r"A:\Crop-Disease-Detection\dataset\cleaned"
)

# Create the directory
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Create Train, Val, and Test folders
for split in ["Train", "Val", "Test"]:
    (CLEANED_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )

print("Cleaned dataset folders created!")

# Display the folder structure
for folder in CLEANED_DIR.iterdir():
    print(folder)

Cleaned dataset folders created!
A:\Crop-Disease-Detection\dataset\cleaned\Test
A:\Crop-Disease-Detection\dataset\cleaned\Train
A:\Crop-Disease-Detection\dataset\cleaned\Val


In [40]:
# Create a set to track image hashes already selected
seen_hashes = set()

# Store images that will be copied
images_to_keep = []

# Store duplicate images that will be skipped
duplicates_to_skip = []

# Process images in Train, Val, Test order
split_order = ["Train", "Val", "Test"]

for split in split_order:

    split_df = valid_df[
        valid_df["Dataset"] == split
    ]

    for _, row in split_df.iterrows():

        image_hash = row["SHA256"]

        if image_hash not in seen_hashes:

            seen_hashes.add(image_hash)
            images_to_keep.append(row)

        else:

            duplicates_to_skip.append(row)

print("Images selected to keep:", len(images_to_keep))
print("Duplicate images to skip:", len(duplicates_to_skip))
print("Unique image hashes:", len(seen_hashes))

Images selected to keep: 67085
Duplicate images to skip: 33
Unique image hashes: 67085


In [41]:
import shutil
from pathlib import Path

copied_count = 0

for row in images_to_keep:

    # Original image location
    source_path = Path(row["File"])

    # Get split and class name
    split = row["Dataset"]
    class_name = row["Class"]

    # Create destination class folder
    destination_folder = CLEANED_DIR / split / class_name
    destination_folder.mkdir(parents=True, exist_ok=True)

    # Destination image location
    destination_path = destination_folder / source_path.name

    # Copy image without modifying original
    shutil.copy2(source_path, destination_path)

    copied_count += 1

print("Dataset cleaning completed!")
print("Total images copied:", copied_count)
print("Total duplicates skipped:", len(duplicates_to_skip))

Dataset cleaning completed!
Total images copied: 67085
Total duplicates skipped: 33


In [42]:
from collections import Counter

cleaned_counts = {}

for split in ["Train", "Val", "Test"]:

    split_path = CLEANED_DIR / split

    class_counts = {}

    for class_folder in split_path.iterdir():

        if class_folder.is_dir():

            image_count = sum(
                1 for file in class_folder.iterdir()
                if file.is_file()
            )

            class_counts[class_folder.name] = image_count

    cleaned_counts[split] = class_counts

    print(f"\n{split} Dataset")
    print("Number of classes:", len(class_counts))
    print("Total images:", sum(class_counts.values()))

print("\nTotal cleaned images:",
      sum(sum(counts.values()) for counts in cleaned_counts.values()))


Train Dataset
Number of classes: 29
Total images: 53672

Val Dataset
Number of classes: 29
Total images: 12060

Test Dataset
Number of classes: 29
Total images: 1353

Total cleaned images: 67085


In [43]:
from collections import defaultdict
import hashlib

cleaned_hashes = defaultdict(list)

for split in ["Train", "Val", "Test"]:

    split_path = CLEANED_DIR / split

    for class_folder in split_path.iterdir():

        if class_folder.is_dir():

            for image_path in class_folder.iterdir():

                if image_path.is_file():

                    sha256 = hashlib.sha256(
                        image_path.read_bytes()
                    ).hexdigest()

                    cleaned_hashes[sha256].append(
                        image_path
                    )

# Find remaining duplicates
remaining_duplicates = {
    image_hash: paths
    for image_hash, paths in cleaned_hashes.items()
    if len(paths) > 1
}

print("Total unique hashes:", len(cleaned_hashes))
print("Remaining duplicate groups:", len(remaining_duplicates))

Total unique hashes: 67085
Remaining duplicate groups: 0


In [44]:
import pandas as pd
from pathlib import Path

REPORT_DIR = Path(
    r"A:\Crop-Disease-Detection\dataset\reports"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Create summary report
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original images",
        "Valid images",
        "Corrupted images",
        "Duplicate groups",
        "Duplicate copies removed",
        "Cleaned images",
        "Remaining duplicate groups",
        "Number of classes"
    ],
    "Value": [
        len(quality_df),
        len(valid_df),
        (quality_df["Status"] == "Corrupted").sum(),
        len(all_duplicate_summary),
        len(duplicates_to_skip),
        len(cleaned_hashes),
        len(remaining_duplicates),
        29
    ]
})

# Save CSV report
cleaning_summary.to_csv(
    REPORT_DIR / "dataset_cleaning_summary.csv",
    index=False
)

# Save class-wise image counts
class_count_rows = []

for split, class_counts in cleaned_counts.items():
    for class_name, count in class_counts.items():
        class_count_rows.append({
            "Split": split,
            "Class": class_name,
            "Image_Count": count
        })

class_counts_df = pd.DataFrame(class_count_rows)

class_counts_df.to_csv(
    REPORT_DIR / "cleaned_class_distribution.csv",
    index=False
)

print("Cleaning reports saved successfully!")
print(cleaning_summary)

Cleaning reports saved successfully!
                       Metric  Value
0             Original images  67118
1                Valid images  67118
2            Corrupted images      0
3            Duplicate groups     33
4    Duplicate copies removed     33
5              Cleaned images  67085
6  Remaining duplicate groups      0
7           Number of classes     29
